# 01 — Data Understanding

This notebook validates the schema, checks data quality, and builds intuition about what we're working with before any transformation.

**Reference:** Géron Ch.2 — *Get the Data* and *Explore the Data to Gain Insights*.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from price_elasticity.config import load_config
from price_elasticity.data import load_training_frame, profile_dataframe

config = load_config(PROJECT_ROOT / 'configs' / 'project.toml')
df = load_training_frame(config)
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 1. Schema Inspection

In [ ]:
df.dtypes.to_frame('dtype').assign(
    non_null=df.notna().sum(),
    missing_pct=(df.isna().mean() * 100).round(2),
    n_unique=df.nunique()
).sort_values('missing_pct', ascending=False)

## 2. Target Columns — Price and Demand

In [ ]:
price_col = config['data']['price_column'].lower()
demand_col = config['data']['demand_column'].lower()
product_col = config['data']['product_column'].lower()

# Convert to numeric
for col in [price_col, demand_col]:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')

valid = df[(df[price_col] > 0) & (df[demand_col] > 0)].copy()
print(f'Valid rows (price > 0 and demand > 0): {len(valid):,} of {len(df):,} ({100*len(valid)/len(df):.1f}%)')
valid[[price_col, demand_col]].describe().round(2)

## 3. Products — Coverage

In [ ]:
obs_per_product = valid.groupby(product_col).agg(
    observations=(demand_col, 'count'),
    distinct_prices=(price_col, 'nunique'),
    avg_price=(price_col, 'mean'),
    avg_demand=(demand_col, 'mean'),
).reset_index()

min_obs = config['modeling']['min_observations']
min_pts = config['modeling']['min_price_points']
eligible = obs_per_product[
    (obs_per_product['observations'] >= min_obs) &
    (obs_per_product['distinct_prices'] >= min_pts)
]

print(f'Total unique products: {obs_per_product[product_col].nunique():,}')
print(f'Eligible for elasticity estimation: {len(eligible):,} '
      f'(>= {min_obs} obs, >= {min_pts} distinct prices)')
obs_per_product.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

obs_per_product['observations'].clip(upper=100).hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].axvline(min_obs, color='red', linestyle='--', label=f'Min threshold ({min_obs})')
axes[0].set_title('Observations per Product (capped at 100)')
axes[0].set_xlabel('Observations')
axes[0].set_ylabel('Number of products')
axes[0].legend()

obs_per_product['distinct_prices'].clip(upper=30).hist(bins=30, ax=axes[1], color='coral', edgecolor='white')
axes[1].axvline(min_pts, color='red', linestyle='--', label=f'Min threshold ({min_pts})')
axes[1].set_title('Distinct Price Points per Product (capped at 30)')
axes[1].set_xlabel('Distinct Prices')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

valid[price_col].clip(upper=valid[price_col].quantile(0.99)).hist(
    bins=60, ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Price Distribution (99th pct cap)')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Frequency')

np.log1p(valid[price_col]).hist(bins=60, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Log(1+Price) Distribution')
axes[1].set_xlabel('log(1 + price)')

plt.suptitle('Price distributions — raw vs log scale', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Sample Records

In [ ]:
# Show one product's full history to understand the granularity
sample_product = eligible.sort_values('distinct_prices', ascending=False).iloc[0][product_col]
print(f'Sample product: {sample_product}')
valid[valid[product_col] == sample_product][[product_col, price_col, demand_col]].sort_values(price_col)

## 6. Data Contract Summary

| Dimension | Value |
|-----------|-------|
| Total rows | See output above |
| Valid rows | price > 0 and demand > 0 |
| Key columns | `name` (product), `disc_price` (price), `imp_count` (demand) |
| Granularity | One row per product-date observation |
| Eligible products | ≥ 12 obs AND ≥ 3 distinct price points |

**Next:** Notebook `02_exploratory_analysis.ipynb` — visualise price/demand relationships and test hypotheses.